<a href="https://colab.research.google.com/github/io-uty/skt-pytorch/blob/main/PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np

# Tensors

In [ ]:
# 자동미분 없이 sin(x)함수를 다항식 y = a + bx + cx² + dx³ 로 근사하는 방법

# 데이터 준비

dtype = torch.float
device = torch.device("cpu")

x = torch.linspace(-torch.pi, torch.pi, 2000, device = device, dtype = dtype)
y = torch.sin(x)

# 파라미터 초기화

a = torch.randn((), device=device, dtype = dtype) # () : 스칼라 의미
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

# 학습 루프 (2000 epoch)
learning_rate = 1e-6
for i in range(2000):
  # 예측값
  y_pred = a+b*x+c*x**2+d*x**3

  # loss 계산 : MSE 의 합
  loss = (y_pred - y).pow(2).sum().item() # .item() → 텐서를 순수 Python 숫자로 변환 (출력용)
  if i%100 == 99: # 100번 반복마다 진행상황 확인
    print(i, loss)

  #Backward pass : 수동으로 gradient 계산
  #chain rule로 각 파라미터에 대한 gradient를 구하는 과정
  grad_y_pred = 2.0 *(y_pred-y)     # dL/dy_pred
  grad_a = grad_y_pred.sum()        # dL/da
  grad_b = (grad_y_pred*x).sum()    # dL/db
  grad_c = (grad_y_pred*x**2).sum() # dL/dc
  grad_d = (grad_y_pred*x**3).sum() # dl/dd

  #update Parameter
  a -= learning_rate * grad_a
  b -= learning_rate * grad_b
  c -= learning_rate * grad_c
  d -= learning_rate * grad_d

#테일러 전개 값과 유사하면 잘 수행된 것
print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 4601.1767578125
199 3047.987548828125
299 2020.2181396484375
399 1340.0946044921875
499 890.000732421875
599 592.1199951171875
699 394.9650573730469
799 264.4681091308594
899 178.0860595703125
999 120.90170288085938
1099 83.04308319091797
1199 57.976898193359375
1299 41.37913513183594
1399 30.387691497802734
1499 23.108285903930664
1599 18.286718368530273
1699 15.092706680297852
1799 12.976644515991211
1899 11.574560165405273
1999 10.645393371582031
Result: y = -0.009144298732280731 + 0.8160338401794434 x + 0.0015775427455082536 x^2 + -0.0875401645898819 x^3


# Autograd

In [ ]:
#Autograd를 사용한 다항식 근사 (exp(x) 근사)

import torch
import math

# 디바이스(Accelerator) 자동 선택
# 사용 가능한 가속기가 있으면 자동으로 그걸 쓰고, 없으면 CPU 사용
dtype = torch.float
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Using {device} device")
torch.set_default_device(device)

# 데이터 및 목표 함수 준비
x = torch.linspace(-1,1,2000,dtype=dtype)
y = torch.exp(x)

# requires_grad=True 가 핵심
# 이전 코드와 가장 큰 차이점
# '이 텐서에 대한 gradient를 추적하고 계산해줘'라고 PyTorch에게 알려주는 것
a = torch.randn((), dtype = dtype, requires_grad = True)
b = torch.randn((), dtype = dtype, requires_grad = True)
c = torch.randn((), dtype = dtype, requires_grad = True)
d = torch.randn((), dtype = dtype, requires_grad = True)

initial_loss=1.
learning_rate =1e-5
for i in range(5000):

# PyTorch가 연산 과정을 자동으로 기록하는 중임
  y_pred = a+b*x + c*x ** 2 + d * x **3

  loss = (y_pred - y).pow(2).sum()

  #상대적 Loss 출력
  #첫 iteration의 loss를 저장해두고, 이후 loss를 초기 loss 대비 비율로 출력
  if i==0:
    initial_loss = loss.item()

  if i%100 == 99:
    print(f'Iteration t = {i:4d}  loss(i)/loss(0) = {round(loss.item()/initial_loss, 6):10.6f}  a = {a.item():10.6f}  b = {b.item():10.6f}  c = {c.item():10.6f}  d = {d.item():10.6f}')

  #PyTorch가 기록해둔 연산 그래프를 따라 역방향으로 체인룰을 자동 적용
  #계산 결과가 각 텐서에 자동으로 저장됨
  loss.backward()

  # torch.no_grad()로 파라미터 업데이트
  # "가중치 업데이트" 자체는 학습 그래프의 일부가 아니라 단순 값 변경이어야 함
  # 따라서 불필요한 추적을 막고 메모리/연산 절약
  with torch.no_grad():
    a -= learning_rate * a.grad
    b -= learning_rate * b.grad
    c -= learning_rate * c.grad
    d -= learning_rate * d.grad

    # Gradient 초기화
    # 매 iteration마다 grad를 초기화하지 않으면
    # 이전 스텝의 gradient가 계속 쌓여서 잘못된 방향으로 학습됨
    a.grad = None
    b.grad = None
    c.grad = None
    d.grad = None
print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

Using cpu device
Iteration t =   99  loss(i)/loss(0) =   0.123919  a =   0.944322  b =  -0.052216  c =   0.663469  d =   1.695477
Iteration t =  199  loss(i)/loss(0) =   0.102770  a =   0.962316  b =   0.065391  c =   0.630310  d =   1.629376
Iteration t =  299  loss(i)/loss(0) =   0.089308  a =   0.971617  b =   0.133185  c =   0.604819  d =   1.540823
Iteration t =  399  loss(i)/loss(0) =   0.077893  a =   0.978343  b =   0.190031  c =   0.586263  d =   1.453804
Iteration t =  499  loss(i)/loss(0) =   0.068041  a =   0.983235  b =   0.242194  c =   0.572766  d =   1.371733
Iteration t =  599  loss(i)/loss(0) =   0.059489  a =   0.986794  b =   0.290841  c =   0.562949  d =   1.294841
Iteration t =  699  loss(i)/loss(0) =   0.052041  a =   0.989382  b =   0.336333  c =   0.555809  d =   1.222879
Iteration t =  799  loss(i)/loss(0) =   0.045541  a =   0.991264  b =   0.378895  c =   0.550615  d =   1.155544
Iteration t =  899  loss(i)/loss(0) =   0.039861  a =   0.992633  b =   0.41871

# Defining new autograd functions


In [ ]:
import torch
import math

class LegendrePolynomial3(torch.autograd.Function):
  """
  We can implement our own custom autograd Functions by subclassing
  torch.autograd.Function and implementing the forward and backward passes
  which operate on Tensors.
  """

  @staticmethod
  # forward — 순전파 계산
  def forward(input):
    """
    In the forward pass we receive a Tensor containing the input and return
    a Tensor containing the output. Check out `Extending torch.autograd <https://docs.pytorch.org/docs/stable/notes/extending.html#extending-torch-autograd>`_
    for further details.
    """
    return 0.5 * (5 * input ** 3 - 3 * input)

  # setup_context — backward에 필요한 값 저장
  @staticmethod
  def setup_context(ctx, inputs, output):
    """
    Store input for use in the backward pass using ``ctx.save_for_backward``.
    Other objects can be stored directly as attributes on the ctx object,
    such as ``ctx.my_object = my_object``.
    """
    input, = inputs
    ctx.save_for_backward(input)

  #  backward — 역전파 계산 (직접 미분 유도)
  @staticmethod
  def backward(ctx, grad_output):
    """
    In the backward pass we receive a Tensor containing the gradient of the loss
    with respect to the output, and we need to compute the gradient of the loss
    with respect to the input.
    """

    input, = ctx.saved_tensors
    return grad_output * 1.5 * (5 * input ** 2 - 1)

dtype = torch.float
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

x = torch.linspace(-math.pi, math.pi, 2000, device = device, dtype =dtype)
y = torch.sin(x)

#torch.full((), 값)으로 특정 값에 고정 초기화
a = torch.full((), 0.0, device=device, dtype=dtype, requires_grad=True)
b = torch.full((), -1.0, device=device, dtype=dtype, requires_grad=True)
c = torch.full((), 0.0, device=device, dtype=dtype, requires_grad=True)
d = torch.full((), 0.3, device=device, dtype=dtype, requires_grad=True)

learning_rate = 5e-6
for t in range(2000):
  P3 = LegendrePolynomial3.apply

  y_pred = a+b*P3(c+d*x)

  # 학습 루프 — 나머지는 이전 코드와 완전히 동일
  loss = (y_pred - y).pow(2).sum()
  if t % 100 == 99:
    print(t, loss.item())

  loss.backward()

  with torch.no_grad():
    a -= learning_rate * a.grad
    b -= learning_rate * b.grad
    c -= learning_rate * c.grad
    d -= learning_rate * d.grad

    a.grad = None
    b.grad = None
    c.grad = None
    d.grad = None

print(f'Result: y = {a.item()} + {b.item} * P3({c.item()} + {d.item()} x)')

99 209.95834350585938
199 144.66018676757812
299 100.70249938964844
399 71.03519439697266
499 50.978511810302734
599 37.403133392333984
699 28.206867218017578
799 21.97318458557129
899 17.7457275390625
999 14.877889633178711
1099 12.93176555633545
1199 11.610918045043945
1299 10.71425724029541
1399 10.10548210144043
1499 9.692105293273926
1599 9.411375999450684
1699 9.220745086669922
1799 9.091285705566406
1899 9.003361701965332
1999 8.943641662597656
Result: y = -6.71270206087371e-10 + <built-in method item of Tensor object at 0x7bbee85cb5c0> * P3(-3.392665037793563e-10 + 0.2554861009120941 x)


# NN

In [ ]:
import torch
import math

x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

#입력 데이터를 (x, x², x³) 형태로 만들기
p = torch.tensor([1, 2, 3])
xx = x.unsqueeze(-1).pow(p)

# nn.Sequential로 모델 정의
model = torch.nn.Sequential( #여러 레이어(Module)를 순서대로 통과시키는 컨테이너
    torch.nn.Linear(3, 1),
    torch.nn.Flatten(0, 1)
)

#nn.MSELoss — 미리 정의된 손실 함수 사용
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-6
for t in range(2000):
  y_pred = model(xx)

  loss = loss_fn(y_pred, y)
  if t % 100 == 99:
      print(t, loss.item())

  #model.zero_grad() — gradient 초기화가 더 간편해짐
  model.zero_grad()
  loss.backward()

  with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad

linear_layer = model[0]
print(f'Result: y = {linear_layer.bias.item()} + {linear_layer.weight[:, 0].item()} x + {linear_layer.weight[:, 1].item()} x^2 + {linear_layer.weight[:, 2].item()} x^3')


99 173.34896850585938
199 119.82958221435547
299 83.79243469238281
399 59.504398345947266
499 43.11943054199219
599 32.0552864074707
699 24.576595306396484
799 19.516441345214844
899 16.08901596069336
999 13.765052795410156
1099 12.187583923339844
1199 11.115636825561523
1299 10.386405944824219
1399 9.889768600463867
1499 9.551155090332031
1599 9.32001781463623
1699 9.162064552307129
1799 9.053997039794922
1899 8.979973793029785
1999 8.929215431213379
Result: y = -0.00865972600877285 + 0.8502115607261658 x + 0.0014939472312107682 x^2 + -0.0924016535282135 x^3
